# Abstract & Introduction

## Abstract

This study investigates the early prediction of discharge capacity in Nickel-Metal Hydride (Ni-MH) cells within an industrial battery-reconditioning context. As the first generation of hybrid vehicles exceeds fifteen years of service life, reconditioning facilities face growing throughput pressure: certifying each cell currently requires a full galvanostatic discharge cycle of up to 14 hours, followed, when needed, by a deep-reconditioning protocol that can extend to 72 additional hours. The central goal of this work is to determine whether machine-learning models can reliably predict the final dischargeable capacity of a cell from a short, partial observation of its discharge telemetry, enabling early classification decisions and reducing effective test time.

To address this challenge, the raw three-dimensional time series (Voltage, Current, Cumulative Capacity) is expanded into a high-dimensional feature representation via discrete finite differences, temporal lag operators, and rolling statistical moments. A sliding-window augmentation strategy mitigates the data-scarcity constraint inherent to the industrial setting. Two parallel predictive tasks are pursued: direct binary classification and continuous capacity regression, and three model architectures of increasing complexity are evaluated: an $\ell_2$-regularized Logistic Regression, a Random Forest ensemble, and a Multi-Layer Perceptron (MLP). All models are assessed under a GroupK-Fold cross-validation protocol enforcing strict cell-level data isolation to prevent leakage between correlated windows.


## Introduction

Measuring the residual capacity of a used battery cell requires, in the most straightforward approach, running the cell to complete discharge and integrating the output current over time. Every cell occupies a test station for up to 14 hours before a pass/fail decision can be made. When a cell requires deep reconditioning to recover from voltage depression, the total cycle time approaches 72–90 hours.

This project investigates the following core question: **can the final dischargeable capacity of a Ni-MH cell be reliably predicted from an observation of the first few minutes of its discharge curve?** A positive answer would enable early-exit test strategies, reducing average station occupancy and significantly increasing facility throughput.

### Research Questions

This project is structured around three explicit, falsifiable research questions:

> **RQ1 — Early Prediction Feasibility.** Do the first $W$ minutes of a Ni-MH discharge curve contain sufficient information to predict its final capacity, as measured by a cross-validated ROC-AUC $> 0.65$ across all GroupK-Fold splits?

> **RQ2 — Feature Engineering vs. Raw Signal.** Does the engineered feature representation (lag operators, discrete derivatives, rolling statistics) yield meaningfully higher predictive performance than training on raw voltage and capacity values alone?

> **RQ3 — Classification vs. Regression.** Does optimizing a continuous regression objective (MSE on $C_{\mathrm{final}}$) and deriving a binary label via post-hoc thresholding outperform a direct binary classifier trained on the discrete label, or vice versa? This question probes whether the continuous capacity continuum contains exploitable structure that a direct classifier discards.

### Notebook Structure

The remainder of this notebook is organized as follows.                                               Section 2 (*Context & Industrial Motivation*) describes the reconditioning workflow and the electrochemical properties of Ni-MH cells. Section 3 (*Problem Formulation*) provides a formal mathematical definition of both prediction tasks. Section 4 (*Exploratory Data Analysis*) characterizes the dataset and validates the key physical assumptions. Section 5 (*Methodology*) details the feature engineering and augmentation pipeline. Section 6 (*Experimental Setup & Model Architectures*) presents the theoretical foundations and implementation of each model. Section 7 (*Results & Discussion*) reports cross-validated performance and tests the research hypotheses. Section 8 (*Conclusions*) summarizes the findings and discusses limitations and future work.


## Mathematical Context & Physical System Dynamics

### Ni-MH Curvature Information and the Flat Plateau Challenge
From a physical and thermodynamic perspective, the choice of Nickel-Metal Hydride (Ni-MH) chemistry dictates unique constraints on the learning algorithms. Unlike other chemistries (such as Lithium-ion variants) that exhibit a quasi-linear voltage decay correlated with the State of Charge, Ni-MH cells possess a remarkably flat electrochemical potential plateau governed by a dual-phase insertion mechanism. The voltage remains locked near the nominal 1.2V threshold for up to 80% of the discharge support domain before exhibiting a sudden, non-linear terminal knee drop. Consequently, during the initial observation window ($W = 10\text{ minutes}$), the absolute voltage variation is infinitesimally small and heavily masked by measurement noise.

# Problem Formulation

## Formal Definition & Dual Prediction Paradigms

Let $\mathcal{D} = \{\mathcal{S}_i\}_{i=1}^{N}$ be a dataset consisting of $N$ independent discharge tests on individual Ni-MH cells. Each test $\mathcal{S}_i$ is a multivariate time series sampled at uniform discrete time steps $t \in \{1, \dots, T_i\}$:

$$\mathcal{S}_i = \left\{ \mathbf{x}_{t,i} \right\}_{t=1}^{T_i} = \left\{ (V_{t,i}, I_{t,i}, C_{t,i}) \right\}_{t=1}^{T_i}$$

where $\mathbf{x}_{t,i} \in \mathbb{R}^3$ is the instantaneous observation vector containing terminal Voltage ($V$, in V), discharge Current ($I$, in A), and cumulative Capacity ($C$, in Ah). The global target quantity is the total capacity delivered before the cutoff voltage is reached:

$$C_{\mathrm{final}, i} = \max_{t \in [1, T_i]} (C_{t,i}) = \sum_{t=1}^{T_i} I_{t,i} \cdot \Delta t$$

To comprehensively evaluate the predictive potential of early discharge observations, two parallel learning tasks are formalized over the same dataset:

1. **Direct Classification.** A function $f_{\text{class}}: \mathbb{R}^{3 \times W} \to \{0, 1\}$ maps the truncated input sequence to a binary label, defined by thresholding the continuous target at a quality criterion $C_{\min}$:
   $$y_i = \mathbb{I}(C_{\mathrm{final}, i} \ge C_{\min})$$
   where $y_i = 1$ denotes a cell with sufficient residual capacity (PASS) and $y_i = 0$ denotes a degraded cell (FAIL). The classifier is trained directly on this discrete label, optimizing a binary cross-entropy loss.

2. **Direct Regression.** A function $f_{\text{reg}}: \mathbb{R}^{3 \times W} \to \mathbb{R}^+$ maps the same truncated input to a continuous capacity estimate $\hat{C}_{\mathrm{final}, i}$, trained to minimize a mean squared error objective. A classification decision can then be derived post-hoc:
   $$\hat{y}_{i}^{(\text{reg})} = \mathbb{I}(\hat{C}_{\mathrm{final}, i} \ge C_{\min})$$

These two paradigms are related rather than independent: the regression output induces a classifier via thresholding, and the two approaches may differ in how well they exploit the continuous structure of the capacity distribution. This relationship is the subject of **RQ3** (stated in the Introduction): whether optimizing over the full capacity continuum yields a better binary decision than optimizing the binary boundary directly.


## The Partial Observation Constraint
Let $W \in \mathbb{N}^+$ denote an informational boundary step such that $W \ll T_i$ for all $i$. The input space accessible to the predictive models is strictly confined to the subset of the trajectory preceding this temporal horizon:
$$\mathbf{X}_{\text{input}, i} = \{\mathbf{x}_{t,i}\}_{t=1}^{W}$$

The objective is to discover and evaluate two distinct hypotheses, $f_{\text{class}} \in \mathcal{H}_{\text{class}}$ and $f_{\text{reg}} \in \mathcal{H}_{\text{reg}}$, mapping the truncated input sequence to their respective domains without any information from $t > W$ leaking into the training phase:
$$f_{\text{class}}: \mathbb{R}^{3 \times W} \to \{0, 1\}, \quad f_{\text{reg}}: \mathbb{R}^{3 \times W} \to \mathbb{R}^+$$


## Statistical Evaluation Metrics
Performance is quantified via distinct loss functionals across the two target domains to evaluate alignment and reliability:
* **Classification Performance Metrics:** Quantified via Empirical Accuracy $\mathbb{P}(Y = \hat{Y})$, Precision $\mathbb{P}(Y = 1 \mid \hat{Y} = 1)$ to isolate Type I errors, Recall, and the Area Under the ROC Curve (ROC-AUC). These metrics are applied to both the direct classifier and the classification derived from the regression threshold.
* **Regression Performance Metrics:** Quantified via Mean Squared Error (MSE), Mean Absolute Error (MAE), and the Coefficient of Determination ($R^2$), evaluating the percentage of variance in the ultimate capacity explained by the early trajectory kinematics:
  $$R^2 = 1 - \frac{\sum_{i} (C_{\mathrm{final}, i} - \hat{C}_{\mathrm{final}, i})^2}{\sum_{i} (C_{\mathrm{final}, i} - \bar{C}_{\mathrm{final}})^2}$$


## Modeling Assumptions

The following assumptions delimit the scope of the study and must be acknowledged when interpreting the results:

* **Uniform sampling rate:** The acquisition frequency is assumed to be constant across all tests, so that a window of $W$ time steps corresponds to the same physical duration $\Delta t \cdot W$ for every cell.
* **Homogeneous initial conditions:** Cells are assumed to arrive at the test station in a fully charged state ($SoC \approx 1$) and under a nominally constant discharge current. Variability in initial state of charge or applied current is not modeled.
* **Environmental stationarity:** Ambient temperature fluctuations during the test are assumed to be negligible or implicitly reflected in the observable voltage dynamics. No explicit temperature signal is available in the dataset.
* **Cell-level independence:** Different test files correspond to independent physical cells. No temporal correlation between files is assumed, and no cell appears more than once in the dataset.


# Exploratory Data Analysis


In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from scipy import stats

# ── Plotting defaults (publication quality) ──────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

RAW_DIR = "data/raw_csvs"

def load_raw_csv(file_path: str) -> pd.DataFrame:
    """Load a single discharge CSV, parse timestamps, and sort chronologically."""
    df = pd.read_csv(file_path, sep=';')
    df['Tempo'] = pd.to_datetime(df['Tempo'], utc=True)
    df = df.sort_values('Tempo').reset_index(drop=True)
    df['elapsed_min'] = (df['Tempo'] - df['Tempo'].iloc[0]).dt.total_seconds() / 60
    df['file_name'] = os.path.basename(file_path)
    return df

# Load all raw files
csv_files = sorted(glob.glob(os.path.join(RAW_DIR, "*.csv")))
print(f"Total test files found: {len(csv_files)}")

all_raw = [load_raw_csv(f) for f in csv_files]

# Dataset summary statistics
summary = pd.DataFrame([{
    "file": os.path.basename(f),
    "n_samples":   len(df),
    "duration_min": df['elapsed_min'].max(),
    "V_min":   df['Tensione (V)'].min(),
    "V_max":   df['Tensione (V)'].max(),
    "C_final": df['Capacity (Ah)'].max(),
} for f, df in zip(csv_files, all_raw)])

print(f"\nDataset summary (N={len(summary)} tests):")
summary.describe().round(3)


## Visualizing Discharge Curves



In [ ]:
# ── Figure 1: Full discharge curves colored by final capacity ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Compute a global capacity range for consistent colormap normalization
c_finals = summary['C_final'].values
norm = plt.Normalize(vmin=c_finals.min(), vmax=c_finals.max())
cmap = cm.RdYlGn  # Red (low capacity) → Green (high capacity)

for df, c_final in zip(all_raw, c_finals):
    color = cmap(norm(c_final))
    axes[0].plot(df['elapsed_min'], df['Tensione (V)'], color=color, alpha=0.55, linewidth=0.8)
    axes[1].plot(df['elapsed_min'], df['Capacity (Ah)'], color=color, alpha=0.55, linewidth=0.8)

# Colorbar
sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
fig.colorbar(sm, ax=axes, label='Final Capacity $C_{\\mathrm{final}}$ (Ah)', fraction=0.02, pad=0.04)

axes[0].set(xlabel='Elapsed Time (min)', ylabel='Voltage (V)',
            title='Fig. 1a — Discharge Voltage Curves')
axes[1].set(xlabel='Elapsed Time (min)', ylabel='Capacity (Ah)',
            title='Fig. 1b — Cumulative Capacity Curves')

plt.tight_layout()
plt.savefig("data/processed/fig1_discharge_curves.png", bbox_inches='tight')
plt.show()
print(f"Capacity range: [{c_finals.min():.3f}, {c_finals.max():.3f}] Ah")
print(f"Duration range: [{summary['duration_min'].min():.1f}, {summary['duration_min'].max():.1f}] min")


## Statistical Properties & Constraints


### Target Variable Distribution


### Feature Correlation Structure


In [ ]:
# ── Figure 2: Final capacity distribution & target label definition ───────────
C_MIN = summary['C_final'].median()  # Threshold at median for balanced classes
summary['label'] = (summary['C_final'] >= C_MIN).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram of final capacity
axes[0].hist(summary['C_final'], bins=15, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(C_MIN, color='crimson', linestyle='--', linewidth=2, label=f'$C_{{\\min}}$ = {C_MIN:.3f} Ah (median)')
axes[0].set(xlabel='Final Capacity $C_{\\mathrm{final}}$ (Ah)',
            ylabel='Count',
            title='Fig. 2a — Distribution of Final Capacity')
axes[0].legend()

# Class balance
class_counts = summary['label'].value_counts().sort_index()
axes[1].bar(['FAIL (y=0)', 'PASS (y=1)'], class_counts.values,
            color=['#e74c3c', '#2ecc71'], edgecolor='white', width=0.5)
for i, v in enumerate(class_counts.values):
    axes[1].text(i, v + 0.2, str(v), ha='center', fontweight='bold')
axes[1].set(ylabel='Number of test files',
            title=f'Fig. 2b — Class Balance (threshold = {C_MIN:.3f} Ah)')

plt.tight_layout()
plt.savefig("data/processed/fig2_target_distribution.png", bbox_inches='tight')
plt.show()

print(f"Threshold C_min = {C_MIN:.4f} Ah")
print(f"Class distribution: PASS={class_counts[1]}, FAIL={class_counts[0]}")
print(f"Class imbalance ratio: {class_counts[1]/class_counts[0]:.2f}")

# ── Kolmogorov–Smirnov normality test on C_final ──────────────────────────────
ks_stat, ks_p = stats.kstest(summary['C_final'], 'norm',
                              args=(summary['C_final'].mean(), summary['C_final'].std()))
print(f"\nKolmogorov–Smirnov test for normality of C_final: D={ks_stat:.4f}, p={ks_p:.4f}")
print("→ Distribution is " + ("approximately normal." if ks_p > 0.05 else "NOT normal (p < 0.05)."))


In [ ]:
# ── Figure 3: Early voltage at t=W vs. final capacity (visual separability) ───
W_MINUTES = 10  # early observation window

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

early_stats = []
for df, label, fname in zip(all_raw, summary['label'], summary['file']):
    early = df[df['elapsed_min'] <= W_MINUTES]
    if len(early) > 5:
        early_stats.append({
            'file': fname,
            'label': label,
            'V_mean_early':  early['Tensione (V)'].mean(),
            'V_std_early':   early['Tensione (V)'].std(),
            'dV_early':      early['Tensione (V)'].iloc[-1] - early['Tensione (V)'].iloc[0],
            'C_at_W':        early['Capacity (Ah)'].iloc[-1],
        })
early_df = pd.DataFrame(early_stats)

palette = {0: '#e74c3c', 1: '#2ecc71'}
labels_str = {0: 'FAIL', 1: 'PASS'}

for ax, xcol, xlabel in zip(axes,
    ['V_mean_early', 'dV_early', 'C_at_W'],
    [f'Mean $V$ in first {W_MINUTES} min (V)',
     f'$\\Delta V$ in first {W_MINUTES} min (V)',
     f'Capacity at $t={W_MINUTES}$ min (Ah)']):
    for lbl in [0, 1]:
        subset = early_df[early_df['label'] == lbl]
        ax.scatter(subset[xcol], subset['C_at_W'] if xcol != 'C_at_W' else
                   [0]*len(subset) + np.random.uniform(-0.01, 0.01, len(subset)),
                   color=palette[lbl], label=labels_str[lbl], alpha=0.7, s=50)
    ax.set(xlabel=xlabel, title=f'Fig. 3 — Separability: {xlabel[:30]}')
    ax.legend()

plt.suptitle(f'Fig. 3 — Linear separability of PASS/FAIL from early ({W_MINUTES}-min) signal features',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig("data/processed/fig3_early_separability.png", bbox_inches='tight')
plt.show()

# Point-biserial correlation test: do early features correlate with the label?
print("Point-biserial correlation with binary label y:")
for col in ['V_mean_early', 'V_std_early', 'dV_early', 'C_at_W']:
    r, p = stats.pointbiserialr(early_df['label'], early_df[col])
    print(f"  {col:20s}: r = {r:+.4f}, p = {p:.4f} {'*' if p < 0.05 else ''}")


# Methodology

This section outlines the formalized framework adopted to transform raw, high-frequency physical measurements into a robust, high-dimensional representation suitable for predictive modeling. Given the constraints of an industrialized laboratory environment, where the total number of independent experimental test cycles is inherently limited ($N \in [63, 80]$), the methodology explicitly focuses on maximizing the information density of each temporal sequence while ensuring strict statistical validation.

## Feature Engineering

Raw telemetry acquired from the battery test bench consists of a low-dimensional time series containing only three core physical variables: Voltage ($V_t$), Current ($I_t$), and cumulative capacity ($C_t$), sampled over Time ($t$). To enable non-sequential models to capture the underlying electro-chemical dynamics, the dataset is mapped into a high-dimensional feature space. 

Instead of relying on raw values, the matrix is expanded at each timestamp $t$ using three primary transformations:

1. **Temporal Lag Features:** To embed short-term memory into the instantaneous state, temporal lags are generated for each variable up to a depth of $k=4$:
   $$X_{\text{lag}} = \{V_{t-1}, \dots, V_{t-4}, C_{t-1}, \dots, C_{t-4}, I_{t-1}, \dots, I_{t-4}\}$$

2. **Discrete Numerical Derivatives:** The kinematics of the discharge profile (the progression from the voltage plateau to the knee drop) are captured via first and second-order finite differences with respect to the actual elapsed time ($dt$):
   * **Voltage Derivatives (Velocity and Acceleration of potential drop):**
     $$\frac{dV}{dt} \approx \frac{V_t - V_{t-1}}{t_t - t_{t-1}}, \quad \frac{d^2V}{dt^2} \approx \frac{\left(\frac{dV}{dt}\right)_t - \left(\frac{dV}{dt}\right)_{t-1}}{t_t - t_{t-1}}$$
   * **Capacity Derivatives:** $\frac{dC}{dt}$ and $\frac{d^2C}{dt^2}$. While $\frac{dC}{dt}$ physically correlates with the charge flow (current), its numerical computation scales the model’s awareness toward sensor quantization noise and hardware fluctuations.

3. **Rolling Statistical Metrics & Physical Interactions:** Local signal smoothing and stability are captured via a moving window ($\omega = 5$) computing the rolling mean and standard deviation of the voltage ($\mu_{\text{roll}}(V), \sigma_{\text{roll}}(V)$). Furthermore, domain-specific physical coupling is introduced by calculating instantaneous Power ($P_t = V_t \times I_t$).

The resulting transformation expands the initial 3-dimensional profile into a comprehensive $\mathbb{R}^{d}$ feature vector ($d \ge 20$) per timestamp.

## Data Augmentation & Windowing

Training advanced architectures like Multi-Layer Perceptrons on fewer than 100 global instances inevitably triggers catastrophic overfitting. To regularize the models and enforce generalization, a **Time-Domain Slicing (Windowing)** data augmentation strategy is implemented.

Rather than treating a multi-hour discharge cycle as a single data point, each continuous CSV sequence is partitioned into multiple sub-sequences using a sliding window of fixed duration $W$ (e.g., 10 minutes) and a displacement step $S$ (e.g., 5 minutes). 

This approach serves a dual academic and practical purpose:
* **Sample Multiplication:** It mathematically scales the available training instances from the initial dozens of complete curves to hundreds of heavily documented, independent windows.
* **Invariance and Early Prediction:** By enforcing a uniform window size across different stages of the discharge process, the models are trained to be time-invariant. This prepares the system for *real-time operational inference*, enabling the prediction of a cell's ultimate behavior by observing any arbitrary window of the initial test, without waiting for the full cycle completion.

In [1]:
from src.data_processing import build_augmented_dataset

# We implement the data enrichment and enhancement pipeline
df_augmented = build_augmented_dataset(
    raw_data_dir="data/raw_csvs",
    output_dir="data/processed",
    lags=4,
    rolling_window=5,
    window_size_min=10,   # Length of the observed window
    step_size_min=5,     # Scroll step (50% overlap)
    sampling_rate_sec=1  # To change based on capacimeter frequence
)

# Visualization of the structure of a high-dimensional matrix
print(f"Generated  columns: {list(df_augmented.columns)}")
df_augmented.head()

Starting Data Augmentation on 63 test files...
Data Augmentation completed successfully!
Resulting global matrix: 3362400 rows, 25 columns.
File saved at: data/processed\dataset_augmented.parquet
Generated  columns: ['file_name', 'time', 'current_t', 'voltage_t', 'capacity_t', 'voltage_t-1', 'capacity_t-1', 'current_t-1', 'voltage_t-2', 'capacity_t-2', 'current_t-2', 'voltage_t-3', 'capacity_t-3', 'current_t-3', 'voltage_t-4', 'capacity_t-4', 'current_t-4', 'dV_dt', 'd2V_dt2', 'dC_dt', 'd2C_dt2', 'voltage_rolling_mean', 'voltage_rolling_std', 'power_t', 'window_id']


,file_name,time,current_t,voltage_t,capacity_t,voltage_t-1,capacity_t-1,current_t-1,voltage_t-2,capacity_t-2,...,capacity_t-4,current_t-4,dV_dt,d2V_dt2,dC_dt,d2C_dt2,voltage_rolling_mean,voltage_rolling_std,power_t,window_id
0,misure_test_131_BAT-02226.csv,2026-03-31 21:08:39+00:00,0.35,8.394,0.000398,8.394,0.000299,0.35,8.395,0.000199,...,0.000000,0.349,0.000,0.001,0.000099,-1.000000e-06,8.3946,0.000548,2.93790,misure_test_131_BAT-02226.csv_w0
1,misure_test_131_BAT-02226.csv,2026-03-31 21:08:40+00:00,0.35,8.393,0.000497,8.394,0.000398,0.35,8.394,0.000299,...,0.000099,0.350,-0.001,-0.001,0.000099,0.000000e+00,8.3942,0.000837,2.93755,misure_test_131_BAT-02226.csv_w0
2,misure_test_131_BAT-02226.csv,2026-03-31 21:08:41+00:00,0.35,8.394,0.000597,8.393,0.000497,0.35,8.394,0.000398,...,0.000199,0.350,0.001,0.002,0.000100,1.000000e-06,8.3940,0.000707,2.93790,misure_test_131_BAT-02226.csv_w0
3,misure_test_131_BAT-02226.csv,2026-03-31 21:08:42+00:00,0.35,8.393,0.000696,8.394,0.000597,0.35,8.393,0.000497,...,0.000299,0.350,-0.001,-0.002,0.000099,-1.000000e-06,8.3936,0.000548,2.93755,misure_test_131_BAT-02226.csv_w0
4,misure_test_131_BAT-02226.csv,2026-03-31 21:08:43+00:00,0.35,8.393,0.000796,8.393,0.000696,0.35,8.394,0.000597,...,0.000398,0.350,0.000,0.001,0.000100,1.000000e-06,8.3934,0.000548,2.93755,misure_test_131_BAT-02226.csv_w0


## Validation Strategy

When dealing with augmented time series derived from a limited set of source files, standard validation techniques like random train-test splitting or standard K-Fold cross-validation introduce a critical flaw known as **Data Leakage**. Because overlapping windows (e.g., Window 1 at $t_{0-10}$ and Window 2 at $t_{5-15}$) originate from the exact same physical battery cell, a random split would place highly correlated samples into both the training and validation sets, yielding artificially inflated, ungeneralizable performance metrics.



To guarantee absolute statistical rigor and mirror a true production deployment, a **GroupK-Fold Cross-Validation** strategy is mandated. 

Under this framework:
* The data partition is strictly grouped by the metadata attribute `file_name` (representing the unique independent test cycle).
* The dataset is split into $K=5$ distinct folds. In any given fold, all augmented windows belonging to a specific battery test file are allocated *either* entirely to the training set *or* entirely to the validation set.

Consequently, the model is always validated on completely unseen battery cells, ensuring that the measured accuracy truly reflects the system's ability to generalize to new hardware in the laboratory.

In [2]:
from sklearn.model_selection import GroupKFold

# Isoliamo la matrice X escludendo le colonne di tracciabilità dell'infrastruttura
exclude_cols = ['file_name', 'window_id', 'time']
feature_cols = [col for col in df_augmented.columns if col not in exclude_cols]

X = df_augmented[feature_cols]
groups = df_augmented['file_name']  # Definiamo i vincoli di gruppo basati sul file originario

# Inizializziamo il GroupKFold accademico
gkf = GroupKFold(n_splits=5)

print("Verifica empirica dell'isolamento dei Folds (No Data Leakage):")
for fold, (train_idx, val_idx) in enumerate(gkf.split(X, groups=groups)):
    train_files = set(df_augmented.iloc[train_idx]['file_name'].unique())
    val_files = set(df_augmented.iloc[val_idx]['file_name'].unique())
    
    # Controllo formale: l'intersezione dei set di file deve essere rigorosamente vuota
    leakage = train_files.intersection(val_files)
    print(f"Fold {fold} -> Campioni Train: {len(train_idx)} | Campioni Val: {len(val_idx)} | File di sovrapposizione: {len(leakage)}")

Verifica empirica dell'isolamento dei Folds (No Data Leakage):
Fold 0 -> Campioni Train: 2690400 | Campioni Val: 672000 | File di sovrapposizione: 0
Fold 1 -> Campioni Train: 2687400 | Campioni Val: 675000 | File di sovrapposizione: 0
Fold 2 -> Campioni Train: 2690400 | Campioni Val: 672000 | File di sovrapposizione: 0
Fold 3 -> Campioni Train: 2689800 | Campioni Val: 672600 | File di sovrapposizione: 0
Fold 4 -> Campioni Train: 2691600 | Campioni Val: 670800 | File di sovrapposizione: 0


# Experimental Setup & Model Architectures


## Baseline: $\ell_2$-Regularized Logistic Regression


### Theoretical Foundation


### $\ell_2$ Regularization (Ridge Penalty)


### Role in This Study


## Machine Learning Alternative: Random Forest


### Theoretical Foundation


### Decision Tree Splitting Criterion


### Variance Reduction via Bagging


### Out-of-Bag (OOB) Error Estimation


### Feature Importance


### Role in This Study


## Deep Learning Framework: Multi-Layer Perceptron (MLP)


### Theoretical Foundation


### Activation Functions


### Training: Backpropagation & Stochastic Gradient Descent


### Regularization Strategies


### The Bias-Variance Tradeoff in the Small-$N$ Regime


### Architecture and Hyperparameters


# Results & Discussion


## Model Evaluation & Comparison


## Analysis of Overfitting & Hyperparameter Sensitivity


## Empirical Proof on Random Forest: OOB Error vs. Number of Trees


## Feature Importance Analysis


# Conclusions


## Summary of Findings


### Hypothesis Verification


## Limitations


## Future Work


## Operational Integration


# References
